# 📈 Notebook 04 — Visualisasi Data & Laporan Insight E-Commerce

**Mata Kuliah**: Data Science  
**Tema**: E-Business  
**Dataset**: E-Commerce Shopee Indonesia (Kaggle)  
**Periode**: Desember 2023 – November 2025

---

Notebook ini menyajikan **9 visualisasi utama** beserta interpretasi dan insight bisnis yang dapat digunakan untuk pengambilan keputusan.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from utils import set_style, fmt_rupiah
set_style()

PALETTE = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51",
           "#023e8a", "#0077b6", "#0096c7", "#00b4d8", "#48cae4"]

# Load data bersih
df = pd.read_csv('../data/processed/all_months_cleaned.csv', low_memory=False)
df['Waktu Pesanan Dibuat'] = pd.to_datetime(df['Waktu Pesanan Dibuat'], errors='coerce')

os.makedirs('../reports/figures', exist_ok=True)
print(f'✅ Data loaded | Shape: {df.shape}')
print(f'   Periode: {df["Waktu Pesanan Dibuat"].min().strftime("%B %Y")} – {df["Waktu Pesanan Dibuat"].max().strftime("%B %Y")}')

---
## Visualisasi 1: Tren Penjualan Bulanan 💰

In [ ]:
# Agregasi per bulan
monthly = df.groupby('Nama_Bulan').agg(
    Jumlah_Transaksi=('order_id', 'count'),
    Total_Pendapatan=('Total Pembayaran', 'sum')
).sort_index()

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Grafik 1: Total Pendapatan
axes[0].plot(monthly.index, monthly['Total_Pendapatan'],
             marker='o', color='#2a9d8f', linewidth=2.5, markersize=6)
axes[0].fill_between(monthly.index, monthly['Total_Pendapatan'],
                     alpha=0.1, color='#2a9d8f')
axes[0].set_title('Tren Total Pendapatan Bulanan', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Total Pendapatan (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Tambahkan anotasi nilai tertinggi
max_idx = monthly['Total_Pendapatan'].idxmax()
max_val = monthly['Total_Pendapatan'].max()
axes[0].annotate(f'{fmt_rupiah(max_val)}',
    xy=(max_idx, max_val),
    xytext=(0, 15), textcoords='offset points',
    ha='center', fontsize=9, color='#e76f51', fontweight='bold')

# Grafik 2: Jumlah Transaksi
axes[1].bar(monthly.index, monthly['Jumlah_Transaksi'],
            color='#264653', alpha=0.8, edgecolor='white')
axes[1].set_title('Tren Jumlah Transaksi Bulanan', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Jumlah Transaksi')
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../reports/figures/04_tren_bulanan.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_tren_bulanan.png')

print(f'\n📊 Insight:')
print(f'   Bulan pendapatan tertinggi : {max_idx} (Rp {max_val:,.0f})')
print(f'   Bulan transaksi terbanyak  : {monthly["Jumlah_Transaksi"].idxmax()} ({monthly["Jumlah_Transaksi"].max():,})')
print(f'   Total pendapatan keseluruhan: Rp {monthly["Total_Pendapatan"].sum():,.0f}')

---
## Visualisasi 2: Distribusi Status Pesanan 📦

In [ ]:
status_counts = df['Status Pesanan'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(status_counts))]
bars = axes[0].barh(status_counts.index, status_counts.values,
                    color=colors_bar, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, status_counts.values):
    axes[0].text(bar.get_width() + max(status_counts.values)*0.01, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Distribusi Status Pesanan', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Jumlah Pesanan')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    status_counts.values,
    labels=status_counts.index,
    autopct='%1.1f%%',
    colors=PALETTE[:len(status_counts)],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Proporsi Status Pesanan', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/04_status_pesanan.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_status_pesanan.png')

total = len(df)
print(f'\n📊 Insight:')
for s, c in status_counts.items():
    print(f'   {s:<30}: {c:,} ({c/total*100:.1f}%)')

---
## Visualisasi 3: Metode Pembayaran 💳

In [ ]:
payment = df['Metode Pembayaran'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top metode pembayaran (bar)
top_payment = payment.head(8)
bars = axes[0].barh(top_payment.index[::-1], top_payment.values[::-1],
                    color=PALETTE[:len(top_payment)], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, top_payment.values[::-1]):
    axes[0].text(bar.get_width() + max(top_payment.values)*0.01,
                bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Top 8 Metode Pembayaran\n(berdasarkan jumlah transaksi)', fontweight='bold')
axes[0].set_xlabel('Jumlah Transaksi')

# Pie chart top 5
top5 = payment.head(5)
others = payment.iloc[5:].sum()
pie_data = pd.concat([top5, pd.Series({'Lainnya': others})])

wedges, texts, autotexts = axes[1].pie(
    pie_data.values,
    labels=pie_data.index,
    autopct='%1.1f%%',
    colors=PALETTE[:len(pie_data)],
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    explode=[0.05] * len(pie_data)
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('Proporsi Metode Pembayaran', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/04_metode_pembayaran.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_metode_pembayaran.png')

print(f'\n📊 Insight:')
print(f'   Metode terpopuler: {payment.idxmax()} ({payment.max():,} transaksi, {payment.max()/total*100:.1f}%)')

---
## Visualisasi 4: Top 10 Provinsi Transaksi 🗺️

In [ ]:
province = df.groupby('Provinsi').agg(
    Jumlah_Transaksi=('order_id', 'count'),
    Total_Pendapatan=('Total Pembayaran', 'sum')
).sort_values('Jumlah_Transaksi', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Jumlah transaksi per provinsi
bars1 = axes[0].barh(
    province.index[::-1], province['Jumlah_Transaksi'][::-1],
    color=PALETTE[:10], alpha=0.85, edgecolor='white'
)
for bar, val in zip(bars1, province['Jumlah_Transaksi'][::-1]):
    axes[0].text(bar.get_width() + max(province['Jumlah_Transaksi'])*0.01,
                bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
axes[0].set_title('Top 10 Provinsi\n(Jumlah Transaksi)', fontweight='bold')
axes[0].set_xlabel('Jumlah Transaksi')

# Total pendapatan per provinsi
province_rev = province.sort_values('Total_Pendapatan', ascending=False)
bars2 = axes[1].barh(
    province_rev.index[::-1], province_rev['Total_Pendapatan'][::-1],
    color='#0077b6', alpha=0.8, edgecolor='white'
)
axes[1].set_title('Top 10 Provinsi\n(Total Pendapatan)', fontweight='bold')
axes[1].set_xlabel('Total Pendapatan (Rp)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))

plt.tight_layout()
plt.savefig('../reports/figures/04_provinsi_top10.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_provinsi_top10.png')

print(f'\n📊 Insight:')
print(f'   Provinsi transaksi terbanyak : {province["Jumlah_Transaksi"].idxmax()}')
print(f'   Provinsi pendapatan tertinggi: {province["Total_Pendapatan"].idxmax()}')

---
## Visualisasi 5: Top 15 Kategori Produk 🛍️

In [ ]:
category = df['product_categories'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(category))]
bars = ax.barh(category.index[::-1], category.values[::-1],
               color=colors[::-1], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, category.values[::-1]):
    ax.text(bar.get_width() + max(category.values)*0.005,
            bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Top 15 Kategori Produk Terlaris', fontsize=14, fontweight='bold')
ax.set_xlabel('Jumlah Transaksi')
ax.set_ylabel('Kategori Produk')
plt.tight_layout()
plt.savefig('../reports/figures/04_kategori_produk.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_kategori_produk.png')

print(f'\n📊 Insight:')
print(f'   Kategori terlaris: {category.idxmax()} ({category.max():,} transaksi)')
print(f'   Total kategori unik: {df["product_categories"].nunique()}')

---
## Visualisasi 6: Distribusi Nilai Transaksi 📊

In [ ]:
# Segmentasi nilai transaksi
bins = [0, 10000, 25000, 50000, 100000, 250000, 500000, float('inf')]
labels = ['<10K', '10K-25K', '25K-50K', '50K-100K', '100K-250K', '250K-500K', '>500K']
df['Segmen_Nilai'] = pd.cut(df['Total Pembayaran'], bins=bins, labels=labels, right=True)

segmen = df['Segmen_Nilai'].value_counts().reindex(labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram detail
tp_trimmed = df['Total Pembayaran'].dropna()
tp_trimmed = tp_trimmed[tp_trimmed < tp_trimmed.quantile(0.99)]
axes[0].hist(tp_trimmed, bins=40, color='#264653', alpha=0.75, edgecolor='white', density=True)
tp_trimmed.plot.kde(ax=axes[0], color='#e76f51', linewidth=2.5, label='KDE')
axes[0].set_title('Histogram Distribusi Total Pembayaran\n(≤ P99)', fontweight='bold')
axes[0].set_xlabel('Total Pembayaran (Rp)')
axes[0].set_ylabel('Densitas')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))
axes[0].legend()

# Segmentasi
colors_seg = PALETTE[:len(segmen)]
bars = axes[1].bar(segmen.index, segmen.values, color=colors_seg, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, segmen.values):
    if not pd.isna(val):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(segmen.dropna())*0.01,
                    f'{val:,.0f}', ha='center', fontsize=9)
axes[1].set_title('Segmentasi Nilai Transaksi', fontweight='bold')
axes[1].set_xlabel('Segmen Nilai')
axes[1].set_ylabel('Jumlah Pesanan')

plt.tight_layout()
plt.savefig('../reports/figures/04_distribusi_nilai.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_distribusi_nilai.png')

---
## Visualisasi 7: Opsi Pengiriman 🚚

In [ ]:
# Ambil nama layanan pengiriman (sebelum '-')
df['Layanan_Kirim'] = df['Opsi Pengiriman'].str.split('-').str[0].str.strip()
shipping = df['Layanan_Kirim'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
bars = axes[0].barh(shipping.index[::-1], shipping.values[::-1],
                    color=PALETTE[:len(shipping)], alpha=0.85, edgecolor='white')
axes[0].set_title('Top 10 Opsi Pengiriman', fontweight='bold')
axes[0].set_xlabel('Jumlah Pesanan')

# Rata-rata ongkir per layanan
avg_ongkir = df.groupby('Layanan_Kirim')['Perkiraan Ongkos Kirim'].mean().sort_values(ascending=False).head(10)
bars2 = axes[1].barh(avg_ongkir.index[::-1], avg_ongkir.values[::-1],
                     color='#e9c46a', alpha=0.85, edgecolor='white')
axes[1].set_title('Rata-rata Ongkos Kirim per Layanan', fontweight='bold')
axes[1].set_xlabel('Rata-rata Ongkir (Rp)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))

plt.tight_layout()
plt.savefig('../reports/figures/04_opsi_pengiriman.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_opsi_pengiriman.png')

---
## Visualisasi 8: Rata-rata Nilai Transaksi per Metode Pembayaran 💵

In [ ]:
avg_payment = df.groupby('Metode Pembayaran').agg(
    Rata_rata_Nilai=('Total Pembayaran', 'mean'),
    Median_Nilai=('Total Pembayaran', 'median'),
    Jumlah_Transaksi=('order_id', 'count')
).sort_values('Rata_rata_Nilai', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(avg_payment))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], avg_payment['Rata_rata_Nilai'],
               width, label='Rata-rata', color='#2a9d8f', alpha=0.85, edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], avg_payment['Median_Nilai'],
               width, label='Median', color='#e9c46a', alpha=0.85, edgecolor='white')

ax.set_title('Rata-rata & Median Nilai Transaksi\nper Metode Pembayaran', fontweight='bold')
ax.set_ylabel('Nilai Transaksi (Rp)')
ax.set_xticks(list(x))
ax.set_xticklabels(avg_payment.index, rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/04_nilai_per_pembayaran.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_nilai_per_pembayaran.png')

---
## Visualisasi 9: Analisis Retur & Pembatalan ↩️

In [ ]:
# Pesanan yang ada retur
df['Ada_Retur'] = df['total_returned_qty'] > 0
retur_pct = df['Ada_Retur'].mean() * 100

# Pesanan dibatalkan
df['Dibatalkan'] = df['Status Pesanan'].str.contains('Batal|Cancel', case=False, na=False)
batal_pct = df['Dibatalkan'].mean() * 100

# Rate retur per provinsi (top 10)
retur_per_prov = df.groupby('Provinsi')['Ada_Retur'].mean().sort_values(ascending=False).head(10) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Donut chart: retur vs tidak
sizes_retur = [retur_pct, 100 - retur_pct]
axes[0].pie(sizes_retur,
            labels=['Ada Retur', 'Tidak Ada Retur'],
            autopct='%1.1f%%',
            colors=['#e76f51', '#2a9d8f'],
            wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            startangle=90)
axes[0].set_title(f'Proporsi Pesanan dengan Retur\n(Total: {df["Ada_Retur"].sum():,} pesanan)', fontweight='bold')

# Rate retur per provinsi
bars = axes[1].barh(retur_per_prov.index[::-1], retur_per_prov.values[::-1],
                    color='#f4a261', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, retur_per_prov.values[::-1]):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)
axes[1].set_title('Rate Retur per Provinsi (Top 10)', fontweight='bold')
axes[1].set_xlabel('Persentase Retur (%)')

plt.tight_layout()
plt.savefig('../reports/figures/04_analisis_retur.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan: 04_analisis_retur.png')

---
## 📋 Ringkasan Insight & Rekomendasi Bisnis

In [ ]:
# Cetak ringkasan lengkap
print('=' * 60)
print('   LAPORAN ANALISIS E-COMMERCE SHOPEE INDONESIA')
print('   Periode: Desember 2023 – November 2025')
print('=' * 60)

print(f'\n📦 VOLUME BISNIS:')
print(f'   Total Transaksi    : {len(df):,}')
print(f'   Total Pendapatan   : Rp {df["Total Pembayaran"].sum():,.0f}')
print(f'   Rata-rata/Transaksi: Rp {df["Total Pembayaran"].mean():,.0f}')
print(f'   Median Transaksi   : Rp {df["Total Pembayaran"].median():,.0f}')

status_selesai = df['Status Pesanan'].value_counts().get('Selesai', 0)
print(f'\n✅ KEPUASAN PELANGGAN:')
print(f'   Pesanan Selesai    : {status_selesai:,} ({status_selesai/len(df)*100:.1f}%)')
print(f'   Pesanan Retur      : {df["Ada_Retur"].sum():,} ({retur_pct:.1f}%)')
print(f'   Pesanan Dibatalkan : {df["Dibatalkan"].sum():,} ({batal_pct:.1f}%)')

print(f'\n💳 PEMBAYARAN:')
top_pay = df['Metode Pembayaran'].value_counts()
print(f'   Metode terpopuler  : {top_pay.idxmax()} ({top_pay.max():,} transaksi)')

print(f'\n🗺️ GEOGRAFIS:')
top_prov = df['Provinsi'].value_counts()
print(f'   Provinsi terbanyak : {top_prov.idxmax()} ({top_prov.max():,} transaksi)')

print(f'\n🛍️ PRODUK:')
top_cat = df['product_categories'].value_counts()
print(f'   Kategori terlaris  : {top_cat.idxmax()} ({top_cat.max():,} transaksi)')
print(f'   Total kategori unik: {df["product_categories"].nunique()}')

print(f'\n📁 FIGURES TERSIMPAN DI: reports/figures/')
import glob
figs = sorted(glob.glob('../reports/figures/*.png'))
for f in figs:
    print(f'   ✅ {os.path.basename(f)}')

---
## 💡 Rekomendasi Bisnis

Berdasarkan analisis data e-commerce di atas, berikut rekomendasi strategis:

### 1. Fokus Pasar Geografis
- **Perkuat distribusi** di provinsi-provinsi dengan transaksi tertinggi
- **Perluas jangkauan** ke provinsi dengan potensi besar namun transaksi rendah

### 2. Optimasi Metode Pembayaran
- Dorong penggunaan **metode digital** (ShopeePay, transfer bank) untuk mengurangi risiko COD
- Berikan **promo cashback** untuk metode digital guna meningkatkan adopsi

### 3. Manajemen Produk
- **Tingkatkan stok** kategori produk terlaris terutama menjelang periode peak
- **Evaluasi kategori** dengan tingkat pembatalan tinggi

### 4. Strategi Pengiriman
- Optimalkan kerjasama dengan **jasa pengiriman terpopuler**
- Pertimbangkan **subsidi ongkir** untuk meningkatkan konversi pesanan

### 5. Penurunan Tingkat Retur
- Identifikasi **penyebab retur** per kategori produk
- Perbaiki **deskripsi & foto produk** untuk mengurangi mismatch ekspektasi

---

**✅ Analisis selesai.** Semua visualisasi tersimpan di `reports/figures/`.